# Phase 2 — Preprocessing

Rebuilds T1/T2 composites with pixel-level cloud masking (SCL-based),
re-exports them, then validates alignment and produces normalized
visualizations using rasterio locally on the exported files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q earthengine-api geemap rasterio

In [ ]:
import ee

ee.Authenticate(force=True)
ee.Initialize(project='stellar-stream-492412-p9')
print('Earth Engine initialized successfully.')
print('IMPORTANT: confirm the account picker matches your Drive account before continuing.')

In [ ]:
import os, sys

REPO_URL = 'https://github.com/karan02566-prog/delhi-ncr-satellite-change.git'
REPO_DIR = '/content/delhi-ncr-satellite-change'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

## Rebuild composites with pixel-level cloud masking

This uses the SCL band to mask clouds/shadow/snow at the pixel level,
then composites — an improvement over Phase 1's scene-level-only
filtering.

In [ ]:
from src.data.gee_download import (
    get_delhi_ncr_roi,
    get_sentinel2_collection_masked,
    get_collection_metadata,
    get_median_composite,
    export_to_drive,
)

roi = get_delhi_ncr_roi()

t1_masked = get_sentinel2_collection_masked('2024-01-01', '2024-02-15', roi, cloud_threshold=20)
t2_masked = get_sentinel2_collection_masked('2026-01-01', '2026-02-15', roi, cloud_threshold=20)

t1_masked_composite = get_median_composite(t1_masked)
t2_masked_composite = get_median_composite(t2_masked)

print('Masked composites built. Image counts (should match Phase 1):')
print('T1 images:', t1_masked.size().getInfo())
print('T2 images:', t2_masked.size().getInfo())

In [ ]:
t1_task = export_to_drive(t1_masked_composite, 'delhi_ncr_t1_2024_masked', roi)
t2_task = export_to_drive(t2_masked_composite, 'delhi_ncr_t2_2026_masked', roi)

print('T1 masked export started:', t1_task.status())
print('T2 masked export started:', t2_task.status())
print('These will take several minutes. Re-run the next cell to poll status.')

In [ ]:
print('T1:', t1_task.status()['state'])
print('T2:', t2_task.status()['state'])

## Once both exports show COMPLETED, validate locally with rasterio

Update the paths below to match the actual Drive locations once
confirmed (check Drive Storage page for exact filenames if unsure).

In [ ]:
import glob

t1_path = glob.glob('/content/drive/MyDrive/**/delhi_ncr_t1_2024_masked*.tif', recursive=True)
t2_path = glob.glob('/content/drive/MyDrive/**/delhi_ncr_t2_2026_masked*.tif', recursive=True)

print('T1 found at:', t1_path)
print('T2 found at:', t2_path)

In [ ]:
from src.data.preprocess import load_raster_info, validate_alignment

t1_info = load_raster_info(t1_path[0])
t2_info = load_raster_info(t2_path[0])

print('T1 info:', t1_info)
print('T2 info:', t2_info)

alignment = validate_alignment(t1_path[0], t2_path[0])
print('Alignment check:', alignment['checks'])

## Visual sanity check: normalized RGB composites

In [ ]:
import rasterio
import matplotlib.pyplot as plt
from src.data.preprocess import normalize_bands

with rasterio.open(t1_path[0]) as src:
    t1_array = src.read()

with rasterio.open(t2_path[0]) as src:
    t2_array = src.read()

# Bands order from gee_download.BANDS: [B2, B3, B4, B8, B11, B12]
# RGB = B4, B3, B2 -> indices 2, 1, 0
t1_rgb = normalize_bands(t1_array[[2, 1, 0], :, :])
t2_rgb = normalize_bands(t2_array[[2, 1, 0], :, :])

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(t1_rgb.transpose(1, 2, 0))
axes[0].set_title('T1 - Jan-Feb 2024 (cloud-masked)')
axes[0].axis('off')
axes[1].imshow(t2_rgb.transpose(1, 2, 0))
axes[1].set_title('T2 - Jan-Feb 2026 (cloud-masked)')
axes[1].axis('off')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/delhi_ncr_change_detection/phase2_rgb_comparison.png', dpi=150)
plt.show()
print('Saved comparison figure to Drive.')